# Grok-multimodal · FS09-FS10 Video & Audio to text


In [ ]:
import os, json, math, random, time, re, string
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path("/kaggle/working"); FIG=OUT/"figures"; RES=OUT/"results"
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device",device,"gpus",torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}


## FS09 · Video to text


In [ ]:
# Video -> text: sequence of shape frames with motion, temporal pool + classifier/caption
# Video kinds: circle moves right, square moves down, triangle blinks color

def make_video(kind, T=8, size=32):
    frames=[]
    for t in range(T):
        img=np.ones((size,size,3),np.float32)*0.95
        yy,xx=np.mgrid[0:size,0:size]
        if kind=="circle_right":
            cx=int(size*(0.2+0.6*t/(T-1))); cy=size//2; r=size*0.15
            m=(yy-cy)**2+(xx-cx)**2<=r**2; img[m]=(0.9,0.2,0.15)
            label="circle moving right"
        elif kind=="square_down":
            cy=int(size*(0.2+0.6*t/(T-1))); cx=size//2; s=int(size*0.18)
            m=(np.abs(yy-cy)<s)&(np.abs(xx-cx)<s); img[m]=(0.15,0.25,0.85)
            label="square moving down"
        elif kind=="triangle_pulse":
            cy,cx=size//2,size//2
            scale=0.15+0.1*math.sin(2*math.pi*t/T)
            m=(yy>cy-size*scale)&(yy<cy+size*scale*1.2)
            m&=np.abs(xx-cx)<(yy-(cy-size*scale))*0.8
            col=(0.15,0.75,0.25) if t%2==0 else (0.9,0.75,0.1)
            img[m]=col; label="triangle color pulse"
        else:
            raise ValueError(kind)
        frames.append(img)
    return np.stack(frames), label  # [T,H,W,3]

VIDEO_KINDS=["circle_right","square_down","triangle_pulse"]

class VideoEnc(nn.Module):
    def __init__(self, n_cls=3):
        super().__init__()
        self.frame=nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten())
        self.temporal=nn.GRU(32,64,batch_first=True)
        self.head=nn.Linear(64,n_cls)
    def forward(self,x):  # [B,T,3,H,W]
        B,T,C,H,W=x.shape
        f=self.frame(x.reshape(B*T,C,H,W)).reshape(B,T,-1)
        o,_=self.temporal(f)
        return self.head(o[:,-1])

def vid_tensor(kind,size=32):
    fr,_=make_video(kind,size=size)
    return torch.tensor(fr.transpose(0,3,1,2),dtype=torch.float32)  # T,3,H,W

class_to_i={k:i for i,k in enumerate(VIDEO_KINDS)}
xs=[]; ys=[]
for k in VIDEO_KINDS:
    for _ in range(160):
        # jitter: add noise frames
        v=vid_tensor(k)
        v=v+0.03*torch.randn_like(v); v=v.clamp(0,1)
        xs.append(v); ys.append(class_to_i[k])
X=torch.stack(xs); Y=torch.tensor(ys)
perm=torch.randperm(len(X)); ntr=int(0.8*len(X))
Xtr,Ytr,Xva,Yva=X[perm[:ntr]],Y[perm[:ntr]],X[perm[ntr:]],Y[perm[ntr:]]

venc=VideoEnc().to(device)
opt=torch.optim.Adam(venc.parameters(),lr=1e-3)
hist9=[]
for epoch in range(1,25):
    venc.train(); perm=torch.randperm(len(Xtr)); losses=[]; accs=[]
    for i in range(0,len(Xtr),16):
        b=perm[i:i+32]; xb,yb=Xtr[b].to(device),Ytr[b].to(device)
        opt.zero_grad(set_to_none=True)
        lg=venc(xb); loss=F.cross_entropy(lg,yb); loss.backward(); opt.step()
        losses.append(loss.item()); accs.append((lg.argmax(1)==yb).float().mean().item())
    venc.eval()
    with torch.no_grad():
        lg=venc(Xva.to(device)); vacc=(lg.argmax(1)==Yva.to(device)).float().mean().item()
    row={"epoch":epoch,"loss":round(float(np.mean(losses)),4),"train_acc":round(float(np.mean(accs)),4),"val_acc":round(vacc,4)}
    hist9.append(row); print(row)

# captions from class
def video_caption(kind):
    v=vid_tensor(kind).unsqueeze(0).to(device)
    with torch.no_grad():
        pred=VIDEO_KINDS[venc(v).argmax(1).item()]
    _,lab=make_video(pred); return lab, pred

rows9=[]
fig,axes=plt.subplots(3,8,figsize=(12,4.5))
for i,k in enumerate(VIDEO_KINDS):
    fr,gt=make_video(k)
    cap,pred=video_caption(k)
    rows9.append({"input":k,"gt":gt,"pred_label":pred,"pred_caption":cap,"ok":pred==k})
    for t in range(8):
        axes[i,t].imshow(fr[t]); axes[i,t].axis("off")
        if t==0: axes[i,t].set_ylabel(k,fontsize=7)
fig.suptitle("FS09 video frames -> text labels")
fig.tight_layout(); fig.savefig(FIG/"fs09_video.png",dpi=120); plt.close()
acc9=sum(r["ok"] for r in rows9)/len(rows9)
fs09={"stage":"FS09","method":"frame CNN + GRU temporal pool -> video label/caption",
      "history":hist9,"samples":rows9,"clean_acc":acc9,
      "vs_prev":"FS02/03 single image; FS09 models motion over time",
      "figure":"figures/fs09_video.png"}
(RES/"fs09.json").write_text(json.dumps(fs09,indent=2)); PROGRESS["FS09"]="ok"; print("FS09 DONE")


## FS10 · Audio to text


In [ ]:
# Audio -> text: synthetic tones as "spoken digits/words", log-mel spectrogram + CTC-like (simplified CE on sequence)
# Words: beep patterns encode color names roughly via frequency

WORD_FREQ={"red":220,"blue":440,"green":660,"stop":880}
WORDS=list(WORD_FREQ.keys())
wstoi={w:i for i,w in enumerate(WORDS)}

def synth_wav(word, sr=8000, dur=0.5):
    t=np.linspace(0,dur,int(sr*dur),endpoint=False)
    f=WORD_FREQ[word]
    # simple AM envelope
    env=np.sin(np.pi*np.clip(t/dur,0,1))**2
    x=0.6*env*np.sin(2*np.pi*f*t)
    # add mild noise
    x=x+0.05*np.random.randn(len(x))
    return x.astype(np.float32), sr

def logmel(x, sr=8000, n_fft=256, hop=128, n_mels=32):
    # numpy STFT
    w=np.hanning(n_fft)
    frames=[]
    for i in range(0,len(x)-n_fft,hop):
        frame=x[i:i+n_fft]*w
        spec=np.fft.rfft(frame)
        frames.append(np.abs(spec)**2)
    S=np.stack(frames,0)  # [T,F]
    # mel filter approx: group bins
    F=S.shape[1]
    mel=np.zeros((S.shape[0], n_mels),np.float32)
    edges=np.linspace(0,F,n_mels+1).astype(int)
    for m in range(n_mels):
        mel[:,m]=S[:,edges[m]:edges[m+1]].mean(1) if edges[m+1]>edges[m] else 0
    return np.log(mel+1e-6).T  # [n_mels,T]

class ASRNet(nn.Module):
    def __init__(self, n_mels=32, n_cls=4):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.ReLU(),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,None)),  # [B,32,1,T]
        )
        self.gru=nn.GRU(32,64,batch_first=True,bidirectional=True)
        self.fc=nn.Linear(128,n_cls)
    def forward(self,x):  # x [B,1,M,T]
        h=self.cnn(x).squeeze(2).transpose(1,2)  # [B,T,32]
        o,_=self.gru(h)
        # mean pool time -> class (utterance classification; CTC-lite)
        return self.fc(o.mean(1))

# dataset
xs=[]; ys=[]
for w in WORDS:
    for _ in range(100):
        wav,_=synth_wav(w,dur=0.45+0.1*random.random())
        # slight pitch jitter via resample-ish
        mel=logmel(wav)
        # pad/crop T to 24
        M,T=mel.shape
        if T<24: mel=np.pad(mel,((0,0),(0,24-T)))
        else: mel=mel[:,:24]
        xs.append(mel[None]); ys.append(wstoi[w])
X=torch.tensor(np.stack(xs),dtype=torch.float32)
Y=torch.tensor(ys)
perm=torch.randperm(len(X)); ntr=int(0.8*len(X))
Xtr,Ytr,Xva,Yva=X[perm[:ntr]],Y[perm[:ntr]],X[perm[ntr:]],Y[perm[ntr:]]

asr=ASRNet().to(device)
opt=torch.optim.Adam(asr.parameters(),lr=1e-3)
hist10=[]
for epoch in range(1,30):
    asr.train(); perm=torch.randperm(len(Xtr)); losses=[]; accs=[]
    for i in range(0,len(Xtr),64):
        b=perm[i:i+64]; xb,yb=Xtr[b].to(device),Ytr[b].to(device)
        opt.zero_grad(set_to_none=True)
        lg=asr(xb); loss=F.cross_entropy(lg,yb); loss.backward(); opt.step()
        losses.append(loss.item()); accs.append((lg.argmax(1)==yb).float().mean().item())
    asr.eval()
    with torch.no_grad():
        vacc=(asr(Xva.to(device)).argmax(1)==Yva.to(device)).float().mean().item()
    row={"epoch":epoch,"loss":round(float(np.mean(losses)),4),"val_acc":round(vacc,4)}
    hist10.append(row)
    if epoch%3==0: print(row)

def asr_decode(word_true):
    wav,_=synth_wav(word_true)
    mel=logmel(wav)
    if mel.shape[1]<24: mel=np.pad(mel,((0,0),(0,24-mel.shape[1])))
    else: mel=mel[:,:24]
    x=torch.tensor(mel[None,None],dtype=torch.float32,device=device)
    with torch.no_grad():
        pred=WORDS[asr(x).argmax(1).item()]
    return pred

rows10=[]
fig,axes=plt.subplots(2,4,figsize=(10,4))
for i,w in enumerate(WORDS):
    wav,_=synth_wav(w); mel=logmel(wav)
    pred=asr_decode(w)
    rows10.append({"gt":w,"pred":pred,"ok":pred==w})
    axes[0,i].plot(wav[:2000],lw=0.5); axes[0,i].set_title(f"wav:{w}",fontsize=8); axes[0,i].set_xticks([])
    axes[1,i].imshow(mel, aspect="auto", origin="lower"); axes[1,i].set_title(f"pred:{pred}",fontsize=8)
fig.suptitle("FS10 audio -> text (tone words)")
fig.tight_layout(); fig.savefig(FIG/"fs10_asr.png",dpi=120); plt.close()
acc10=sum(r["ok"] for r in rows10)/len(rows10)
fs10={"stage":"FS10","method":"log-mel + CNN/BiGRU utterance ASR (synthetic tones)",
      "history":hist10,"samples":rows10,"acc":acc10,
      "vs_prev":"FS09 video frames; FS10 time-frequency audio pathway to text",
      "figure":"figures/fs10_asr.png"}
(RES/"fs10.json").write_text(json.dumps(fs10,indent=2)); PROGRESS["FS10"]="ok"; print("FS10 DONE",acc10)


In [ ]:
summary={"notebook":"Grok-multimodal-fs09-fs10-video-audio","progress":PROGRESS,"device":str(device)}
(RES/"summary_fs09_fs10.json").write_text(json.dumps(summary,indent=2))
(OUT/"SUCCESS").write_text("ok\n"); print(summary)
